In [16]:
import pandas as pd
import os
import numpy as np
from configs.constants import MCBOOST_NAME

MCBOOST_TABLE_NAME = 'MCGrad'

In [17]:
res_dir = 'mc_industry_results/tuned/'
res_files = os.listdir(res_dir)

In [18]:
dfs = []
for f in res_files:
    if not f.endswith('.pkl'):
        continue
    dfs.append(pd.read_pickle(os.path.join(res_dir, f)))
df = pd.concat(dfs)
df = df[df.seed == 15].copy()
df['mcb_algorithm'] = df.mcb_algorithm.fillna('base_model')

In [23]:
# Categorize the different mcboost variants based on parameter values
def categorize_mcboost(mcb_algorithm, params):
    if mcb_algorithm == 'HKRR':
        return f'HKRR_{params["lambda"]}_{params["alpha"]}'
    if mcb_algorithm != MCBOOST_NAME:
        return mcb_algorithm
    if mcb_algorithm == MCBOOST_NAME:
        return params['name']

    print(params)
    return None

In [24]:
resolved_algos = [categorize_mcboost(row['mcb_algorithm'], row['mcb_algorithm_params']) for _, row in df.iterrows()]

{'name': 'DFMCBoost', 'feature_type': 'group', 'unshrink': False, 'encode_categorical_variables': True, 'num_rounds': 10, 'lightgbm_params': {'max_depth': 2}, 'early_stopping': False, 'tune_hyperparameters': False}
{'name': 'DFMCBoost', 'feature_type': 'group', 'unshrink': False, 'encode_categorical_variables': True, 'num_rounds': 10, 'lightgbm_params': {'max_depth': 2}, 'early_stopping': False, 'tune_hyperparameters': False}
{'name': 'DFMCBoost', 'feature_type': 'group', 'unshrink': False, 'encode_categorical_variables': True, 'num_rounds': 10, 'lightgbm_params': {'max_depth': 2}, 'early_stopping': False, 'tune_hyperparameters': False}
{'name': 'DFMCBoost', 'feature_type': 'group', 'unshrink': False, 'encode_categorical_variables': True, 'num_rounds': 10, 'lightgbm_params': {'max_depth': 2}, 'early_stopping': False, 'tune_hyperparameters': False}
{'name': 'DFMCBoost', 'feature_type': 'group', 'unshrink': False, 'encode_categorical_variables': True, 'num_rounds': 10, 'lightgbm_params':

In [8]:
df['mcb_algorithm'] = resolved_algos

In [9]:
df.mcb_algorithm.value_counts()

mcb_algorithm
base_model         38
HKRR_0.1_0.0125    38
Isotonic           38
MCB                38
MCB_grps_only      38
MCB_no_unshrink    38
MCB_one_round      38
MCB_msh_20         38
DFMCBoost          38
Name: count, dtype: int64

In [46]:
df = df[df.mcb_algorithm.notnull()]

In [47]:
df_test = df[df.set_name == 'test']
### For HKRR take the validation results to find best params per dataset and remove other results from the test results
df_val_hkrr = df[(df.set_name == 'validation') & (df.mcb_algorithm.str.startswith('HKRR')) & (df.group == 'agg')].reset_index()
best_hkrr_params_per_dataset = df_val_hkrr.iloc[df_val_hkrr.groupby('dataset')['logloss'].idxmin()][['dataset', 'mcb_algorithm']].assign(matched=True)
df_test = pd.merge(df_test, best_hkrr_params_per_dataset, on=['dataset', 'mcb_algorithm'], how='left')
df_test['select'] = df_test.apply(lambda row: not np.isnan(row.matched) or not row.mcb_algorithm.startswith('HKRR'), axis=1)
df_test = df_test[df_test.select]
df_test = df_test.drop(columns=['matched', 'select'])
df_test['mcb_algorithm'] = df_test.mcb_algorithm.apply(lambda x: x if not x.startswith('HKRR') else 'HKRR')

In [48]:
# Check if we're left with one result per dataset per algorithm on aggregated test data
(df_test[df_test.group == 'agg'].groupby(['dataset', 'mcb_algorithm']).group.count() != 1).any()

np.False_

In [49]:
res_tab_index_cols = ['dataset', 'model', 'mcb_algorithm']
metric_name_mapping = {
    'MCE_perc': 'MCE_perc_features',
    'MCE_sigma': 'MCE_sigma_features',
    'ECCE_perc': 'global_ECCE_perc',
    'ECCE_sigma': 'global_ECCE_sigma',
    'ECE': 'global_ECE',
    'logloss': 'logloss',
    'prauc': 'prauc',
    'fit_time': 'fit_time',
    'num_rounds': 'num_rounds_mcboost',

}
agg_df = df_test[df_test.group == 'agg']

res_tab = agg_df[res_tab_index_cols + list(metric_name_mapping.keys()) + ['mcb_algorithm_params']].set_index(res_tab_index_cols)
res_tab = res_tab.rename(columns=metric_name_mapping)

max_df = df_test[df_test.group == 'max']
max_df_sel = max_df[res_tab_index_cols + ['ECCE_perc', 'ECCE_sigma', 'smECE']].set_index(res_tab_index_cols)
max_df_sel = max_df_sel.rename(columns={'ECCE_perc': 'MCE_perc_groups', 'ECCE_sigma': 'MCE_sigma_groups', 'smECE': 'max_group_smECE'})
res_tab = res_tab.join(max_df_sel, how='left')

In [50]:
# Check there are no extra rows
(res_tab.reset_index(drop=False).groupby(['dataset', 'mcb_algorithm']).prauc.count() != 1).any()

np.False_

In [51]:
# Take just LogisticRegression
lr_res_tab = res_tab.xs('LogisticRegression', level=1)

In [52]:
pretty_colnames = {
    'global_ECCE_perc': 'ECCE-%',
    'global_ECCE_sigma': 'ECCE-σ',
    'global_ECE': 'ECE',
    'logloss': 'Log-Loss',
    'prauc': 'PRAUC',
    'MCE_perc_features': 'MCE-% (all)',
    'MCE_sigma_features': 'MCE-σ (all)' ,
    'MCE_perc_groups': 'MCE-% (grp)',
    'MCE_sigma_groups': 'MCE-σ (grp)',
    'max_group_smECE': 'max smECE (grp)',
}

In [53]:
lr_res_tab.to_csv('kdd_mcboost_results.csv', index=True)

In [54]:
lr_res_tab

MCE_perc_features  MCE_sigma_features  \
dataset       mcb_algorithm                                            
BankMarketing base_model                 16.1890              7.0101   
              HKRR                       15.0349              6.6617   
              Isotonic                   14.6371              6.3438   
              MCB                         9.1002              4.1535   
              MCB_grps_only              13.4730              5.8165   
              MCB_no_unshrink             8.5084              3.8441   
              MCB_one_round              11.4819              5.0625   
              MCB_msh_20                 12.9567              5.9140   
              DFMCBoost                  13.7458              5.9398   

                               global_ECCE_perc  global_ECCE_sigma  \
dataset       mcb_algorithm                                          
BankMarketing base_model                13.8149             5.9821   
              HKRR                       7.2755             3.2237   
              Isotonic                   2.4898             1.0791   
              MCB                        2.5525             1.1650   
              MCB_grps_only              2.9223             1.2616   
              MCB_no_unshrink            3.0790             1.3911   
              MCB_one_round              6.0604             2.6721   
              MCB_msh_20                 2.8080             1.2817   
              DFMCBoost                  3.8050             1.6442   

                               global_ECE  logloss   prauc  fit_time  \
dataset       mcb_algorithm                                            
BankMarketing base_model           0.0320   3.6076  0.3107       NaN   
              HKRR                 0.0100   3.6235  0.3432  6.399212   
              Isotonic             0.0060   3.5478  0.3531  0.005512   
              MCB                  0.0051   3.4202  0.3718  8.378201   
              MCB_grps_only        0.0039   3.5278  0.3492  4.362778   
              MCB_no_unshrink      0.0053   3.4122  0.3713  8.157368   
              MCB_one_round        0.0126   3.5079  0.3452  1.061012   
              MCB_msh_20           0.0062   3.4082  0.3733  9.929791   
              DFMCBoost            0.0077   3.5238  0.3458  1.063691   

                               num_rounds_mcboost  \
dataset       mcb_algorithm                         
BankMarketing base_model                      NaN   
              HKRR                            NaN   
              Isotonic                        NaN   
              MCB                             7.0   
              MCB_grps_only                   3.0   
              MCB_no_unshrink                 7.0   
              MCB_one_round                   1.0   
              MCB_msh_20                      8.0   
              DFMCBoost                       1.0   

                                                            mcb_algorithm_params  \
dataset       mcb_algorithm                                                        
BankMarketing base_model                                                    None   
              HKRR                              {'lambda': 0.1, 'alpha': 0.0125}   
              Isotonic                                                        {}   
              MCB              {'feature_type': 'features', 'unshrink': True,...   
              MCB_grps_only    {'feature_type': 'group', 'unshrink': True, 'e...   
              MCB_no_unshrink  {'feature_type': 'features', 'unshrink': False...   
              MCB_one_round    {'feature_type': 'features', 'unshrink': True,...   
              MCB_msh_20       {'feature_type': 'features', 'unshrink': True,...   
              DFMCBoost        {'feature_type': 'group', 'unshrink': False, '...   

                               MCE_perc_groups  MCE_sigma_groups  \
dataset       mcb_algorithm                                        
BankMarketing base_model     

In [34]:
global_metrics_res_tab = lr_res_tab[['global_ECCE_perc', 'global_ECCE_sigma', 'global_ECE', 'logloss', 'prauc']].rename(columns=pretty_colnames)
mc_metrics_res_tab = lr_res_tab[['MCE_perc_features', 'MCE_sigma_features', 'MCE_perc_groups', 'MCE_sigma_groups', 'max_group_smECE']].rename(columns=pretty_colnames)

### Latex tables

In [35]:
import pandas as pd
import re

def escape_latex(text):
    """Escape LaTeX special characters in text."""
    if not isinstance(text, str):
        return text
    replacements = {
        '\\': r'\textbackslash{}',
        '&': r'\&',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '_': r'\_',
        '{': r'\{',
        '}': r'\}',
        '~': r'\textasciitilde{}',
        '^': r'\^{}',
    }
    pattern = re.compile('|'.join(re.escape(k) for k in replacements))
    return pattern.sub(lambda m: replacements[m.group()], text)

def multiindex_to_latex_highlighted(df, digits=2):
    """Convert MultiIndex DataFrame to LaTeX tabular format with per-group highlights and lines after each top-level group."""
    higher_is_better = ['prauc']
    numeric_cols = df.select_dtypes(include='number').columns

    group_level = 0
    group_min, group_max = {}, {}

    for name, group in df.groupby(level=group_level):
        for col in numeric_cols:
            if col in higher_is_better:
                group_max[(name, col)] = group[col].max()
            else:
                group_min[(name, col)] = group[col].min()

    latex_lines = []
    last_values = [""] * df.index.nlevels

    headers = [escape_latex(h) for h in list(df.index.names) + list(df.columns)]
    latex_lines.append("\\begin{tabular}{" + "l" * len(headers) + "}")
    latex_lines.append("\\toprule")
    latex_lines.append(" & ".join(headers) + " \\\\")
    latex_lines.append("\\midrule")

    current_group = None
    for idx, row in df.iterrows():
        top_group = idx[group_level]

        # If a new group starts, insert a line before it (but not before the first group)
        if current_group is not None and top_group != current_group:
            latex_lines.append("\\midrule")
        current_group = top_group

        row_items = []
        for level, value in enumerate(idx):
            if value == last_values[level]:
                row_items.append("")
            else:
                row_items.append(escape_latex(str(value)))
                last_values[level] = value
            for reset_level in range(level + 1, df.index.nlevels):
                last_values[reset_level] = ""

        for col in df.columns:
            val = row[col]
            if isinstance(val, float):
                formatted = f"{val:.{digits}f}"
                if col in higher_is_better and val == group_max.get((top_group, col), None):
                    formatted = f"\\textbf{{{formatted}}}"
                elif col not in higher_is_better and val == group_min.get((top_group, col), None):
                    formatted = f"\\textbf{{{formatted}}}"
            else:
                formatted = escape_latex(str(val))
            row_items.append(formatted)

        latex_lines.append(" & ".join(row_items) + " \\\\")

    latex_lines.append("\\bottomrule")
    latex_lines.append("\\end{tabular}")
    return "\n".join(latex_lines)


In [36]:
print(multiindex_to_latex_highlighted(mc_metrics_res_tab))

\begin{tabular}{lllllll}
\toprule
dataset & mcb\_algorithm & MCE-\% (all) & MCE-σ (all) & MCE-\% (grp) & MCE-σ (grp) & max smECE (grp) \\
\midrule
BankMarketing & base\_model & 16.19 & 7.01 & 35.26 & 5.12 & 0.06 \\
 & HKRR & 15.03 & 6.66 & 30.18 & 2.56 & 0.04 \\
 & Isotonic & 14.64 & 6.34 & 23.55 & 2.46 & 0.04 \\
 & MCB & 9.10 & 4.15 & \textbf{19.04} & \textbf{1.87} & 0.03 \\
 & MCB\_grps\_only & 13.47 & 5.82 & 26.91 & 2.12 & 0.04 \\
 & MCB\_no\_unshrink & \textbf{8.51} & \textbf{3.84} & 19.49 & 2.15 & \textbf{0.03} \\
 & MCB\_one\_round & 11.48 & 5.06 & 19.09 & 3.01 & 0.04 \\
 & MCB\_msh\_20 & 12.96 & 5.91 & 21.36 & 1.90 & 0.03 \\
 & DFMCBoost & 13.75 & 5.94 & 26.71 & 2.31 & 0.04 \\
\bottomrule
\end{tabular}


In [37]:
print(multiindex_to_latex_highlighted(global_metrics_res_tab))

\begin{tabular}{lllllll}
\toprule
dataset & mcb\_algorithm & ECCE-\% & ECCE-σ & ECE & Log-Loss & PRAUC \\
\midrule
BankMarketing & base\_model & 13.81 & 5.98 & 0.03 & 3.61 & \textbf{0.31} \\
 & HKRR & 7.28 & 3.22 & 0.01 & 3.62 & 0.34 \\
 & Isotonic & \textbf{2.49} & \textbf{1.08} & 0.01 & 3.55 & 0.35 \\
 & MCB & 2.55 & 1.16 & 0.01 & 3.42 & 0.37 \\
 & MCB\_grps\_only & 2.92 & 1.26 & \textbf{0.00} & 3.53 & 0.35 \\
 & MCB\_no\_unshrink & 3.08 & 1.39 & 0.01 & 3.41 & 0.37 \\
 & MCB\_one\_round & 6.06 & 2.67 & 0.01 & 3.51 & 0.35 \\
 & MCB\_msh\_20 & 2.81 & 1.28 & 0.01 & \textbf{3.41} & 0.37 \\
 & DFMCBoost & 3.81 & 1.64 & 0.01 & 3.52 & 0.35 \\
\bottomrule
\end{tabular}
